In [0]:
%pip install tqdm

In [0]:
import os
import pandas as pd
from tqdm import tqdm

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
CATALOG = "use1_prod_artemis_catalog_3718194974443840" #Change
SCHEMA = "tier1_raw" #Change

flight_table = f"{CATALOG}.{SCHEMA}.drone_mission_table"
TABLE_NAME_CLIP = f"{CATALOG}.{SCHEMA}.drone_plot_clipped_table"

In [0]:
flight_df = spark.read.table(flight_table).toPandas()
len(flight_df)

In [0]:
row_list = []

for idx, row in tqdm(flight_df.iterrows(), total=len(flight_df)):
    flight_path = os.path.dirname(row['flight_metadata_path'])

    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")
    
    # 2. Apuntamos a la carpeta de parcelas recortadas (en minúsculas por convención)
    plot_clipped_path = f"{flight_path}/plot_clipped"
    plot_count = 0
    
    # 3. Contamos las imágenes SIN IMPORTAR si la carpeta existe o no todavía
    if os.path.exists(plot_clipped_path):
        files = os.listdir(plot_clipped_path)
        plot_count = len([f for f in files if f.lower().endswith(('.tif', '.jpg', '.jpeg', '.png'))])

    # 4. Construimos el diccionario SIEMPRE para mantener el inventario completo
    row_dict = {
        'site': row['site'],
        'trial': row['trial'],
        'season': row['season'],
        'field':  row['field'],
        'location': row['location'],
        'mission': row['mission'],
        'flight_date': row['flight_date'],
        
        # Rutas y métricas específicas de esta etapa
        'flight_metadata_path': row['flight_metadata_path'],
        'plot_clipped_path': plot_clipped_path,
        'plots_exist': plot_count > 0, # Bandera equivalente a 'ortho_exists'
        'plot_image_count': plot_count 
    }
    
    row_list.append(row_dict)

# 5. Generamos el DataFrame final
plot_df = pd.DataFrame(row_list)

print(f"Total flights processed for plots inventory: {len(plot_df)}")

# 6. Guardado permanente en Delta Table (adaptado de tu versión anterior)
if len(plot_df) > 0:
    display(plot_df)
    
    # Descomenta estas líneas cuando quieras guardar la tabla en Unity Catalog
    # df_to_save = spark.createDataFrame(plot_df)
    # df_to_save.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME_CLIP)
    # print(f" Processed inventory permanently saved to: {TABLE_NAME_CLIP}")
else:
    print("It's still empty. Check if you ran the flight_df table before running this.")

In [0]:
spark_df = spark.createDataFrame(plot_df)

spark_df.printSchema()

In [0]:
spark_df.write.option("mergeSchema", "true").saveAsTable(TABLE_NAME_CLIP, mode="overwrite")